##Data Frames

####¿Qué es un Data Frame?
> Tabla estructurada con filas y columnas, fácil de usar y optimizada. 

####¿Cómo los creamos?
> Desde archivos CSV, Parquet, bases de datos, etc.

####Ventajas de DataFrames
> •	Optimización automática.   
•	Procesamiento en memoria para mayor velocidad.  



> En entornos como Apache Spark, los DataFrames pueden estar distribuidos en múltiples nodos para procesamiento paralelo y escalable.


In [0]:
from pyspark.sql.functions import *

In [0]:
###Codigo para leer un archivo CSV y cargarlo en un DataFrame
df_1 = spark.read.format("csv").option("header", True).load( 
    "/Volumes/workspace/default/volumen_prueba/info_dummy_em8024 2.csv" #Modifica la ruta, con la ruta de tu archivo
).limit(10)
display(df_1)

In [0]:
df_2 = spark.read.table(
    "samples.bakehouse.sales_transactions"
)
display(df_2.limit(10))

####Seleccionar y añadir columnas

In [0]:
df_3 = df_2.select(
    ["transactionID","dateTime","product", "quantity", "unitprice"]
).orderBy(asc('product'))
display(df_3)

In [0]:
df_3 = df_3.withColumn(
    "total_price",
    df_3["quantity"] * df_3["unitprice"]
)
display(df_3)

In [0]:
####Columna bigSale
df_3 = df_3.withColumn(
    "bigSale",
    df_3 ["total_price"] > 100
)
display(df_3)


Databricks data profile. Run in Databricks to view.

Databricks data profile. Run in Databricks to view.

In [0]:
__dw_df_0_in = df_3
__dw_df_0 = __dw_df_0_in.filter("(quantity > 30)") \
     .filter("(product in ('Austin Almond Biscotti'))")

display(__dw_df_0)

####Filtrar

In [0]:
df_4 = df_3.filter("(product == 'Pearly Pies')")
display(df_4)

In [0]:
####Filtro Automatico

####Ejercicio 1.
> a) Seleccionar solo las columnas transactionID, product, y totalPrice.  
b) Filtrar transacciones pagadas con ‘amex’.   
c) Filtrar transacciones donde la cantidad (quantity) es mayor a 30.  
d) Añade una nueva columna calculando el valor unitario por producto.  

a) Seleccionar solo las columnas transactionID, product, y totalPrice.

In [0]:
df_ejercicio1= df_2.select(["transactionID","product", "totalPrice"]).orderBy(asc('product'))
display(df_ejercicio1)

In [0]:
df_ejercicio1 = df_2.filter("(paymentMethod == 'amex')")
display(df_ejercicio1)

c) Filtrar transacciones donde la cantidad (quantity) es mayor a 30.

In [0]:
df_ejercicio1 = df_2.filter("(quantity > 30)")
display(df_ejercicio1)

d) Añade una nueva columna calculando el valor unitario por producto.

In [0]:
from pyspark.sql.functions import lit

####Ejercicio 1.
df_ejercicio1= df_2.select(["transactionID","product", "totalPrice"]).orderBy(asc('product'))
df_ejercicio1 = df_2.filter("(paymentMethod == 'amex')")
df_ejercicio1 = df_2.filter("(quantity > 30)")



df_ejercicio1 = df_ejercicio1.withColumn("totales", df_2["totalPrice"] * df_2["unitprice"])
display(df_ejercicio1)

### Metodos de agrupación de datos


| Método | Descripción |
| --- | --- |
| agg | Permite calcular varios valores (como suma, promedio, etc.) al mismo tiempo para cada grupo. |
| avg | Calcula el valor medio (promedio) de una columna numérica para cada grupo. |
| count | Cuenta cuántas filas hay en cada grupo. |
| max | Encuentra el valor más alto (máximo) de una columna numérica en cada grupo. |
| min | Encuentra el valor más bajo (mínimo) de una columna numérica en cada grupo. |
| sum | Calcula el total (suma) de una columna numérica para cada grupo. |

In [0]:
df_5 = df_2.groupBy("product").sum("totalPrice")
display(df_5)

In [0]:
###Agrupar por dos columnas y dos metodos de agrupación
df_6 = df_2.groupBy("franchiseID","product").agg(sum("totalPrice").alias("sum_total_price"),avg("totalPrice").alias("avg_total_price"))
display(df_6)

In [0]:
###Escribir tabla en esquema
df_6.write.mode("overwrite").saveAsTable("workspace.default.tbl_total_ventas_agg")

In [0]:
###Escribir tabla usando append

####Ejercicio 2.
> Crea un DataFrame de los productos vendidos y sus totales de solo aquellas transacciones que se pagaron por medio de visa.  

In [0]:
####Ejercicio 2.
df_ejercicio2 = df_2.filter("(paymentMethod == 'visa')")
df_ejercicio2 = df_2.groupBy("product").agg(sum("totalPrice").alias("total_ventas"))
display(df_ejercicio2)



####Ejercicio 3.
> Muestra el producto con mayores ventas pagadas con 'visa'  

In [0]:
####Ejercicio 3.
df_ejercicio3 = df_2.filter("(paymentMethod == 'visa')")
df_ejercicio3 = df_2[df_2['Ganancias'].idxmax()]

display(df_ejercicio3)